# Hari 22 — Evaluation: Meninjau Ulang Seluruh Proyek

**Ini bukan Hari 18/21 lagi.** Hari 18 dan 21 itu evaluasi TEKNIS model (MAE, trend accuracy, cross-validation). Fase **Evaluation** resmi di CRISP-DM itu lebih luas: meninjau ulang **seluruh proyek** — apakah keseluruhan proses (bukan cuma model) benar-benar menjawab business question dari Hari 2, dan apakah kita boleh lanjut ke Deployment atau justru perlu putar balik ke fase sebelumnya. CRISP-DM itu **iteratif**, bukan garis lurus — fase Evaluation adalah titik resmi untuk memutuskan itu.

In [2]:
# Cell ini sudah lengkap — sanity check paling dasar: apakah model_final.pkl dari
# perbaikan Hari 21 benar-benar tersimpan dan bisa dimuat ulang dengan benar.

import joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error

model_final = joblib.load("model_final.pkl")
print(f"Model berhasil dimuat. Tipe: {type(model_final).__name__}")
print(f"Parameter: {model_final.get_params()}")

Model berhasil dimuat. Tipe: RandomForestRegressor
Parameter: {'bootstrap': True, 'ccp_alpha': 0.0, 'criterion': 'squared_error', 'max_depth': 7, 'max_features': 1.0, 'max_leaf_nodes': None, 'max_samples': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 2, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'monotonic_cst': None, 'n_estimators': 100, 'n_jobs': None, 'oob_score': False, 'random_state': 42, 'verbose': 0, 'warm_start': False}


## Uji Kewajaran: Apakah Baseline yang Dipakai Selama Ini Adil?

Ada satu kejanggalan yang perlu diperiksa serius sebelum kita menyatakan proyek ini "berhasil": baseline dari Hari 2 (**MAE 12.953**, dihitung SEKALI dari seluruh histori data) jauh lebih kecil dibanding rata-rata MAE cross-validation model manapun (53 ribu - 73 ribu dari Hari 19-21). Kalau dibaca mentah-mentah, ini kelihatannya seperti model kita justru **jauh lebih buruk** dari baseline saat diuji ke banyak periode waktu — bertentangan dengan kesimpulan "model mengalahkan baseline" yang kita pegang sejak Hari 16.

**Tapi ini perbandingan yang tidak adil.** Baseline naive tidak butuh data training sama sekali — dia langsung bisa jalan dari baris pertama. Sementara fold-fold awal `TimeSeriesSplit` cuma punya ~24-28 baris training untuk model ML — sangat sedikit, bikin model ML kesulitan di fold-fold awal itu secara khusus. Supaya adil, baseline PERLU dihitung ulang **per fold yang sama persis** dengan model ML, baru dibandingkan head-to-head.

In [3]:
# Cell ini sudah lengkap — perbandingan fold-demi-fold yang adil antara baseline dan model.

df_fitur = pd.read_csv("dataset_siap_modeling.csv", index_col=0, parse_dates=True)
X_full = df_fitur.drop(columns=["target_minggu_depan"])
y_full = df_fitur["target_minggu_depan"]

kasus_asli = pd.read_csv("dataset_bersih_minggu2.csv", index_col=0, parse_dates=True)["kasus_baru_mingguan"]
current_actual_full = kasus_asli.loc[X_full.index]

tscv = TimeSeriesSplit(n_splits=5)

baseline_scores, linear_scores, rf_tuned_scores = [], [], []

for train_idx, test_idx in tscv.split(X_full):
    y_test_fold = y_full.iloc[test_idx]
    current_fold = current_actual_full.iloc[test_idx]

    # Baseline naive: prediksi minggu depan = kasus minggu ini (dihitung KHUSUS untuk fold ini)
    baseline_scores.append(mean_absolute_error(y_test_fold, current_fold))

    X_train_fold, X_test_fold = X_full.iloc[train_idx], X_full.iloc[test_idx]
    y_train_fold = y_full.iloc[train_idx]

    scaler_fold = StandardScaler().fit(X_train_fold)
    lin = LinearRegression().fit(scaler_fold.transform(X_train_fold), y_train_fold)
    linear_scores.append(mean_absolute_error(y_test_fold, lin.predict(scaler_fold.transform(X_test_fold))))

    rf = RandomForestRegressor(random_state=42, max_depth=7, min_samples_leaf=2).fit(X_train_fold, y_train_fold)
    rf_tuned_scores.append(mean_absolute_error(y_test_fold, rf.predict(X_test_fold)))

perbandingan_adil = pd.DataFrame({
    "Baseline (per fold)": baseline_scores,
    "Linear Regression": linear_scores,
    "Random Forest (tuned)": rf_tuned_scores,
})
print(perbandingan_adil)
print("\nRata-rata:")
print(perbandingan_adil.mean())

   Baseline (per fold)  Linear Regression  Random Forest (tuned)
0          5503.478261       14714.994072           21050.376624
1         26558.260870       40801.815651           71878.122939
2         11532.173913      282551.498499          110008.706701
3         27308.130435       23350.215870           53903.509267
4          5232.043478        3646.495843            8983.453357

Rata-rata:
Baseline (per fold)      15226.817391
Linear Regression        73013.003987
Random Forest (tuned)    53164.833778
dtype: float64


## Latihan: Interpretasi Hasil Perbandingan Adil

In [4]:
# LATIHAN — tulis kodenya sendiri sesuai panduan komentar di bawah:
# 1. Untuk tiap fold, cek apakah Linear Regression mengalahkan baseline PADA FOLD ITU
#    (bandingkan perbandingan_adil["Linear Regression"] < perbandingan_adil["Baseline (per fold)"]
#    per baris), simpan sebagai kolom baru "Linear Menang?"
# 2. Lakukan hal sama untuk Random Forest tuned, kolom "RF Menang?"
# 3. Cetak perbandingan_adil dengan 2 kolom tambahan itu
# 4. Hitung berapa dari 5 fold masing-masing model berhasil mengalahkan baseline PADA
#    FOLD YANG SAMA (bukan cuma rata-rata keseluruhan)
# Tulis kode kamu di bawah ini:
perbandingan_adil["Linear Menang"] = (
    perbandingan_adil["Linear Regression"] < perbandingan_adil["Baseline (per fold)"])
perbandingan_adil["RF Menang"] = (
    perbandingan_adil["Random Forest (tuned)"] < perbandingan_adil["Baseline (per fold)"]
)

print(perbandingan_adil)
print(f"Linear menang di {perbandingan_adil['Linear Menang'].sum()} dari {len(perbandingan_adil)} fold")
print(f"RF Menang di {perbandingan_adil["RF Menang"].sum()} dari  fold")

   Baseline (per fold)  Linear Regression  Random Forest (tuned)  \
0          5503.478261       14714.994072           21050.376624   
1         26558.260870       40801.815651           71878.122939   
2         11532.173913      282551.498499          110008.706701   
3         27308.130435       23350.215870           53903.509267   
4          5232.043478        3646.495843            8983.453357   

   Linear Menang  RF Menang  
0          False      False  
1          False      False  
2          False      False  
3           True      False  
4           True      False  
Linear menang di 2 dari 5 fold
RF Menang di 0 dari  fold


Angka "berapa dari 5 fold menang" ini jauh lebih jujur dibanding sekadar "MAE rata-rata lebih rendah" — karena kamu bisa lihat **konsistensi** model mengalahkan baseline di berbagai periode waktu, termasuk periode-periode sulit (gelombang awal pandemi, transisi Delta-Omicron), bukan cuma di satu window test yang kebetulan tenang seperti di Hari 16-17.

## Checklist: Kembali ke Project Charter Hari 2

Isi checklist ini berdasarkan SEMUA bukti yang sudah terkumpul (Hari 16-22), jujur — termasuk kalau ternyata ada kriteria yang cuma "terpenuhi sebagian":

| Kriteria (dari Hari 2) | Terpenuhi? | Catatan |
|---|---|---|
| Technical: MAE model < baseline (single-split) | ✅ / ❌ | ✅ |
| Technical: MAE model < baseline (konsisten di cross-validation, tiap fold) | ✅ / ❌ / Sebagian | Sebagian |
| Technical: perbaikan minimal 15-20% dari baseline | ✅ / ❌ | ✅ |
| Business: output bisa diringkas jadi label Naik/Turun/Stabil | ✅ / ❌ | ✅|
| Business: masyarakat awam bisa paham tanpa latar belakang statistik | ✅ / ❌ / Belum diuji ke pengguna asli | Belum diuji ke pengguna asli |
| Scope: prediksi nasional, horizon 1 minggu (sesuai Hari 2) | ✅ / ❌ | ✅ |

## Keterbatasan Proyek yang Sudah Diketahui (Dikumpulkan dari Seluruh 22 Hari)

Bagian penting dari fase Evaluation: mendokumentasikan apa yang **belum sempurna**, bukan menyembunyikannya.

- Tidak ada data vaksinasi sebelum Januari 2021 (diisi 0, dicatat sejak Hari 8-9)
- Fitur "lag-0" (kasus minggu ini sebagai fitur langsung) belum pernah dimasukkan ke model manapun (ditemukan Hari 18)
- `lag_1`, `lag_2`, `lag_3`, `rolling_mean_4w` berkorelasi tinggi — koefisien Linear Regression individual tidak sepenuhnya bisa dipercaya (Hari 16)
- Dataset kecil (143 baris) — fold-fold awal cross-validation punya training set sangat terbatas, performa model kurang stabil di periode-periode itu (Hari 19-22)
- Business success criteria ("masyarakat awam paham") belum pernah diuji ke pengguna sungguhan — baru diuji secara proxy lewat trend accuracy

## Keputusan Go/No-Go untuk Deployment

Isi bagian ini sebagai keputusan resmi fase Evaluation:

> **Keputusan:** Lanjut ke Deployment / Putar balik ke fase sebelumnya (coret salah satu) → Putar balik ke fase sebelumnya
>
> **Alasan:** Tambahkan fitur lag-0. Tidak jadi pakai Random Forest (tuned), pakai Linear Regression yang trend accuarcy-nya tinggi walaupun kurang stabil
>
> **Kalau lanjut ke Deployment — batasan yang harus dijelaskan ke pengguna:** (misal: "model ini kurang andal di periode gelombang tajam, gunakan dengan hati-hati saat tren sedang berubah cepat") → **gunakan dengan hati-hati saat tren sedang berubah cepat**
>
> **Kalau ada kesempatan iterasi lanjutan di luar 30 hari ini, fase mana yang paling prioritas diulang?** (Data Preparation untuk tambah fitur lag-0? Modeling untuk coba Ridge Regression?) → **Data Preparation untuk tambah fitur lag-0**

## Refleksi Hari 22

> 1. Dari perbandingan adil fold-demi-fold — berapa dari 5 fold model final kamu benar-benar mengalahkan baseline? Apakah ini mengubah keyakinanmu terhadap proyek ini? → 0 dari 5 fold (RF tuned), 2 dari 5 (Linear Regression)
> 2. Menurutmu, kriteria mana dari Project Charter Hari 2 yang paling "terpenuhi sebagian" alih-alih terpenuhi penuh? → **MAE model < baseline (konsisten di cross-validation, tiap fold)**
> 3. Apakah kamu memutuskan lanjut ke Deployment? Kalau ya, apa SATU batasan paling penting yang wajib disampaikan ke pengguna aplikasi nanti? → belum lanjut

---
### Selanjutnya: Hari 23-26

- **Hari 23**: Interpretasi lebih dalam model final (feature importance/koefisien), dikaitkan dengan batasan yang baru saja kamu dokumentasikan
- **Hari 24**: Konsep deployment — load `model_final.pkl`, uji coba prediksi manual dengan data terbaru
- **Hari 25-26**: Bangun aplikasi Streamlit — input data terbaru, output prediksi + label tren, DENGAN batasan/disclaimer dari keputusan Go/No-Go hari ini ditampilkan ke pengguna